#

#Installations and Preperation

##Installing Packages

In [ ]:
!pip install -q pyvis


In [ ]:
!pip install -q von-sdk

##Importing Packages

In [ ]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
from geopy.geocoders import Nominatim
from time import sleep as wait
from tqdm import tqdm

import re
import os
import networkx as nx
import pandas as pd
import numpy as np
import math

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
import folium
from folium.plugins import MarkerCluster
import json
import shapely
import networkx as nx
import IPython
from pyvis.network import Network
import von


##Cloning

In [ ]:
%%bash
rm -rf LM4GeoAI Corpus2GeoRAG
git clone https://github.com/EtzionR/Corpus2GeoRAG.git

##Reading NER Data

In [ ]:
df = pd.read_parquet('Corpus2GeoRAG/ApplyNER/NER_output/post_NER_final.parquet')

print(f'Dataframe shape: {df.shape}\n\n')

df.head()


In [ ]:
df.entity_group.value_counts()

#GeoCoding

In [ ]:
loc_entities = list(df[df.entity_group=='LOC'].word.unique())

print(f'{len(loc_entities)} LOC entities found by the NER model\n')

In [ ]:
# Geocoding settings
TARGET_COUNTRIES = {'ua', 'ru'}             # preferred countries (ISO codes, as returned by Nominatim)
REGION_MARGIN = 0.15                        # max importance gap for preferring a UA/RU place / a city over the chosen match
SIMPLIFY_FACTOR = 0.002                     # simplification tolerance = factor * sqrt(area), i.e. ~0.2% of the place's size
SIMPLIFY_RANGE = (0.0005, 0.1)              # tolerance limits, in degrees (~50 m to ~10 km)
# All geocoded locations, in the repo (cloned above); new locations are added to it
GEO_RESULTS_PATH = 'Corpus2GeoRAG/Graph_n_Geocoding/geocoding_results.parquet'
ALLOW_FULL_GEOCODING = False  # True: geocode all LOC entities from scratch (~4 h) when the file is not in the repo
GEO_COLS = ['latitude', 'longitude', 'address', 'place_type', 'importance', 'geometry_wkt']

geolocator = Nominatim(user_agent="GeoAI_Wikipedia_Corpus_to_GeoGraph", timeout=10)

# Wrap geolocator to automatically delay requests (1 sec); errors are raised so they are not saved
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, swallow_exceptions=False)

def get_importance(loc):
    return loc.raw.get('importance') or 0.0

def get_country_code(loc):
    return loc.raw.get('address', {}).get('country_code')

def geocode_entity(word):
    """Geocode one LOC entity, preferring a UA/RU place when it is within REGION_MARGIN of Nominatim's top match."""
    candidates = geocode(word, exactly_one=False, limit=5, language='en', addressdetails=True, geometry='wkt')
    if not candidates:
        return {'word': word, 'geo_reranked': False, 'geo_city_preferred': False}

    # Best UA/RU settlement or administrative area, considered only when the top match is outside UA/RU
    top = candidates[0]
    regional = [c for c in candidates
                if get_country_code(c) in TARGET_COUNTRIES and c.raw.get('class') in ('place', 'boundary')]
    contender = max(regional, key=get_importance) if regional and get_country_code(top) not in TARGET_COUNTRIES else None

    reranked = contender is not None and get_importance(contender) >= get_importance(top) - REGION_MARGIN
    loc = contender if reranked else top

    # Prefer a city/town named exactly like the entity over an oblast/state (e.g. 'Donetsk' -> city, not Donetsk Oblast)
    cities = [c for c in candidates
              if c.raw.get('addresstype') in ('city', 'town') and (c.raw.get('name') or '').lower() == word.lower()
              and get_country_code(c) == get_country_code(loc)]
    city = max(cities, key=get_importance) if cities and loc.raw.get('addresstype') == 'state' else None

    city_preferred = city is not None and get_importance(city) >= get_importance(loc) - REGION_MARGIN
    loc = city if city_preferred else loc

    return {
        'word': word,
        'latitude': loc.latitude,
        'longitude': loc.longitude,
        'address': loc.address,
        'place_type': loc.raw.get('addresstype') or loc.raw.get('type'),
        'importance': loc.raw.get('importance'),
        'geometry': fine_geometry(loc.raw.get('geotext')),  # polygon (or point, when OSM has no polygon), ~50 m
        # Audit fields (used by the validation and re-ranking cells, not added to the graph)
        'country_code': get_country_code(loc),
        'geo_reranked': reranked,
        'geo_city_preferred': city_preferred,
        'top_address': top.address,
        'top_importance': top.raw.get('importance'),
        'regional_address': contender.address if contender else None,
        'regional_importance': contender.raw.get('importance') if contender else None,
    }

def fine_geometry(wkt):
    """Polygon simplified at the smallest tolerance (SIMPLIFY_RANGE[0], ~50 m), as stored in the results file."""
    return shapely.to_wkt(shapely.from_wkt(wkt).simplify(SIMPLIFY_RANGE[0]), rounding_precision=5) if isinstance(wkt, str) else None

def simplify_geometry(wkt):
    """Simplify with a tolerance proportional to the geometry's size (sqrt of its area), clipped to SIMPLIFY_RANGE."""
    if not isinstance(wkt, str):
        return None, None
    geom = shapely.from_wkt(wkt)
    tolerance = min(max(SIMPLIFY_FACTOR * math.sqrt(geom.area), SIMPLIFY_RANGE[0]), SIMPLIFY_RANGE[1])
    return shapely.to_wkt(geom.simplify(tolerance), rounding_precision=5), tolerance


##Geocoding New Locations
Reads `geocoding_results.parquet` from the repo and sends only the LOC entities that are not in it yet to Nominatim (1 request/sec). When there are new ones, the file is updated: upload it to GitHub.

In [ ]:
# Geocoding results from the repo; only LOC entities that are not in it yet are sent to Nominatim
if not os.path.exists(GEO_RESULTS_PATH) and not ALLOW_FULL_GEOCODING:
    raise FileNotFoundError(f'{GEO_RESULTS_PATH} is not in the repo. Pull the latest repo, or set ALLOW_FULL_GEOCODING = True '
                            'to geocode all LOC entities from scratch (~4 h)')
results = pd.read_parquet(GEO_RESULTS_PATH).to_dict('records') if os.path.exists(GEO_RESULTS_PATH) else []

known = {r['word'] for r in results}
new_locations = [w for w in loc_entities if w not in known]
print(f'{len(loc_entities) - len(new_locations)} of {len(loc_entities)} LOC entities already geocoded, '
      f'{len(new_locations)} new (~{len(new_locations) / 3600:.1f} h at 1 request/sec)')

if new_locations:
    for i, word in enumerate(tqdm(new_locations, desc='Geocoding new locations'), 1):
        try:
            results.append(geocode_entity(word))
        except Exception as e:
            tqdm.write(f'Skipped {word!r} ({type(e).__name__}), will retry on next run')
        if i % 100 == 0:  # checkpoint, fast compression
            pd.DataFrame(results).to_parquet(GEO_RESULTS_PATH, index=False, compression='zstd')

    # Final save with strong compression (~2-3 min), small enough for GitHub
    pd.DataFrame(results).to_parquet(GEO_RESULTS_PATH, index=False, compression='zstd', compression_level=19)
    print(f'Updated {GEO_RESULTS_PATH}: {len(results)} locations, {os.path.getsize(GEO_RESULTS_PATH) / 1e6:.1f} MB. '
          'Commit and push it to GitHub (re-running the cloning cell replaces the repo folder)')


##Geocoded Locations

In [ ]:
# Locations of this run: the stored ~50 m polygons are kept for the spatial INSIDE relations (geometry_fine)
# and simplified further by size for the graph (geometry_wkt)
geo_df = pd.DataFrame(results)
geo_df = geo_df[geo_df['word'].isin(loc_entities)].reset_index(drop=True)
geo_df['geometry_fine'] = geo_df.pop('geometry')
geo_df['geometry_wkt'], geo_df['geometry_tolerance'] = zip(*geo_df['geometry_fine'].map(simplify_geometry))

# Merge back into the main DataFrame (drop first, so re-running the cell does not duplicate columns)
df = df.drop(columns=GEO_COLS, errors='ignore').merge(geo_df[['word'] + GEO_COLS], on='word', how='left')


In [ ]:
# Example: the main DataFrame after geocoding, one row for each of the 10 most frequent LOC entities
loc_rows = df[df['entity_group'] == 'LOC']
top_locations = loc_rows['word'].value_counts().head(10).index

example = loc_rows.drop_duplicates('word').set_index('word').loc[
    top_locations, ['entity_group', 'Source', 'start', 'end', 'latitude', 'longitude', 'address', 'place_type', 'importance', 'geometry_wkt']]
example['geometry_wkt'] = example['geometry_wkt'].str[:45] + ' ...'

print(f"{loc_rows['latitude'].notna().sum()} of {len(loc_rows)} LOC rows have coordinates")
with pd.option_context('display.max_colwidth', 70):
    display(example)


In [ ]:

# Filter out entities without valid coordinates
map_data = geo_df.dropna(subset=['latitude', 'longitude'])

# Center map coordinates
center_lat = map_data['latitude'].mean()
center_lon = map_data['longitude'].mean()

# Esri World Street Map tiles
esri_tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}'
esri_attr = 'Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China, TomTom'

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=3,
    tiles=esri_tiles,
    attr=esri_attr
)

# Add marker cluster
marker_cluster = MarkerCluster().add_to(m)

# Add markers with hover tooltips and click popups
for _, row in map_data.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=row['word'],
        popup=f"<b>Entity:</b> {row['word']}<br><b>Address:</b> {row['address']}"
              f"<br><b>Type:</b> {row['place_type']}<br><b>Importance:</b> {row['importance']:.2f}"
    ).add_to(marker_cluster)

# Display interactive map
m


##Polygon Map

In [ ]:
# Entities to draw: the 20 most frequent LOC entities, or set your own list, e.g. ['Donetsk', 'Kyiv', 'Crimea']
POLYGON_ENTITIES = df.loc[df['entity_group'] == 'LOC', 'word'].value_counts().head(20).index.tolist()

poly_data = geo_df[geo_df['word'].isin(POLYGON_ENTITIES) & geo_df['geometry_wkt'].notna()]

m_poly = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles=esri_tiles, attr=esri_attr)

# One toggleable layer per entity, drawn from the simplified geometry stored in the graph
for _, row in poly_data.iterrows():
    folium.GeoJson(
        shapely.from_wkt(row['geometry_wkt']),
        name=row['word'],
        tooltip=f"{row['word']} ({row['place_type']}) | tolerance {row['geometry_tolerance']:.4f}°",
        style_function=lambda _: {'color': '#d32f2f', 'weight': 2, 'fillOpacity': 0.1},
    ).add_to(m_poly)

folium.LayerControl(collapsed=False).add_to(m_poly)

# Display interactive map
m_poly


##Geocoding Validation

In [ ]:
# Nominatim returns WGS 84 geographic coordinates (EPSG:4326) in decimal degrees.
# Axis order: latitude/longitude columns and Folium use (lat, lon); WKT and shapely use (lon lat).
found = geo_df.dropna(subset=['latitude', 'longitude'])

in_range = found['latitude'].between(-90, 90) & found['longitude'].between(-180, 180)
# Simplification moves a boundary by at most its tolerance, so the point must lie within it
on_geometry = found.apply(
    lambda r: shapely.from_wkt(r['geometry_wkt']).distance(shapely.Point(r['longitude'], r['latitude'])) <= r['geometry_tolerance']
    if isinstance(r['geometry_wkt'], str) else True, axis=1)
outside_region = ~found['country_code'].isin(TARGET_COUNTRIES)
wkt_kb = found['geometry_wkt'].str.len() / 1024

print('CRS: WGS 84 (EPSG:4326), decimal degrees\n')
print(f'Geocoded: {len(found)}/{len(geo_df)} ({len(geo_df) - len(found)} not found)')
print(f'Coordinates out of range: {(~in_range).sum()}')
print(f'Point farther from its own geometry than the simplification tolerance: {(~on_geometry).sum()}')
print(f'Outside UA/RU: {outside_region.sum()}')
print(f"Simplified geometries: {wkt_kb.sum() / 1024:.1f} MB in total, largest {wkt_kb.max():.0f} KB ({found.loc[wkt_kb.idxmax(), 'word']})\n")
print(f"Not found (sample): {geo_df.loc[geo_df['latitude'].isna(), 'word'].head(20).tolist()}\n")

print('=== Place Types ===')
print(found['place_type'].value_counts().head(15))

# Rows failing a coordinate check
found[~in_range | ~on_geometry][['word', 'latitude', 'longitude', 'address']]


##Re-ranking Audit

In [ ]:
# 1. Entities whose top match was outside UA/RU while a UA/RU place was among the candidates
audit = geo_df.dropna(subset=['regional_address']).copy()
audit['importance_gap'] = (audit['top_importance'].astype(float) - audit['regional_importance'].astype(float)).round(3)
audit_cols = ['word', 'top_address', 'top_importance', 'regional_address', 'regional_importance', 'importance_gap']

# Sorted so the cases closest to the margin come first
reranked = audit[audit['geo_reranked']].sort_values('importance_gap', ascending=False)
kept = audit[~audit['geo_reranked']].sort_values('importance_gap')

# 2. Entities switched from an oblast/state to the city of the same name
city_preferred = geo_df[geo_df['geo_city_preferred']]

print(f'REGION_MARGIN = {REGION_MARGIN}: {len(reranked)} re-ranked to UA/RU, {len(kept)} kept the top match, '
      f'{len(city_preferred)} switched from oblast/state to city\n')

with pd.option_context('display.max_colwidth', None):
    print('=== Re-ranked to UA/RU (largest gaps first) ===')
    display(reranked[audit_cols].head(15))
    print('\n=== Kept Top Match (smallest gaps first) ===')
    display(kept[audit_cols].head(15))
    print('\n=== Oblast/State -> City ===')
    display(city_preferred[['word', 'top_address', 'top_importance', 'address', 'importance']].head(15))


#Knoledge Graph

##Von

In [ ]:
# Hugging Face token from Colab Secrets (key icon in the Colab sidebar -> secret named HF_TOKEN, notebook access on);
# falls back to a hidden prompt where Secrets are not available
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass('Hugging Face token: ')


In [ ]:
# Relation schemas per entity-type pair, in the canonical order PER -> ORG -> LOC ('a' = first entity, 'b' = second;
# for same-type pairs 'a' is the one mentioned first). Each choice: key -> (relation, subject, description), where
# subject is the entity the relation reads from ('a' / 'b', None = symmetric). Descriptions are filled with the real
# names, and both directions of a directional relation are separate choices, so von also decides the direction.
RELATION_SCHEMAS = {
    'PER_LOC': {
        'BORN_IN': ('BORN_IN', 'a', '{a} was born in, comes from, or is a native of {b}'),
        'LOCATED_IN': ('LOCATED_IN', 'a', '{a} lived, worked, served, fought, was held or was present in {b}'),
        'VISITED': ('VISITED', 'a', '{a} visited, travelled to or arrived in {b}'),
        'FLED_TO': ('FLED_TO', 'a', '{a} fled, escaped, emigrated or was deported to {b}'),
        'DIED_IN': ('DIED_IN', 'a', '{a} died, was killed or was buried in {b}'),
        'LEADS': ('LEADS', 'a', '{a} holds an office of {b}: its president, prime minister, minister, governor, mayor or head of administration'),
        'OPPOSES': ('OPPOSES', 'a', '{a} attacked, ordered an attack on, threatened or acted against {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} (its government or forces) attacked, killed, sanctioned, arrested, banned or accused {a}'),
        'STATED_ABOUT': ('STATED_ABOUT', 'a', '{a} is quoted as saying, claiming or writing something about {b}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'PER_ORG': {
        'MEMBER_OF': ('MEMBER_OF', 'a', '{a} is or was a member, employee, soldier, deputy, researcher, student or official of {b}'),
        'LEADS': ('LEADS', 'a', '{a} leads, heads, commands, chairs or founded {b}, or is its chief, director or deputy head'),
        'OPPOSES': ('OPPOSES', 'a', '{a} opposed, criticized, attacked or fought against {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} accused, criticized, sanctioned, arrested, detained, attacked or killed {a}'),
        'STATED_ABOUT': ('STATED_ABOUT', None, '{b} is cited as the source of a report or claim about {a} ("according to {b}"), or {a} is quoted about {b}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'PER_PER': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same person: another name, spelling, transliteration, title or nickname'),
        'RELATIVE_OF': ('RELATIVE_OF', None, '{a} and {b} are family: married, spouses, parent and child, or siblings'),
        'MET_WITH': ('MET_WITH', None, '{a} and {b} met, talked, phoned, negotiated or reached an agreement with each other'),
        'ALLIED_WITH': ('ALLIED_WITH', None, '{a} and {b} are allies, colleagues or confidants, or worked together or supported each other'),
        'OPPOSES': ('OPPOSES', 'a', '{a} opposed, criticized, attacked or accused {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} opposed, criticized, attacked or accused {a}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'ORG_LOC': {
        'BASED_IN': ('BASED_IN', 'a', '{a} is headquartered, based or stationed in {b}, has its capital in {b}, or was relocated to {b}'),
        'OPERATES_IN': ('OPERATES_IN', 'a', '{a} operates, fought, was deployed or works in {b}'),
        'ATTACKED': ('ATTACKED', 'a', '{a} attacked, shelled, bombed, invaded or struck {b}'),
        'CONTROLS': ('CONTROLS', 'a', '{a} controls, captured, occupied or governs {b}'),
        'SUPPORTS': ('SUPPORTS', None, 'one of {a} and {b} gave aid, weapons, funding or political support to the other'),
        'OPPOSES': ('OPPOSES', None, 'one of {a} and {b} sanctioned, banned, punished, accused or acted against the other'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'ORG_ORG': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same organization: an abbreviation, a former or later name, or {a} became or was renamed {b}'),
        'PART_OF': ('PART_OF', 'a', 'the sentence says {a} is a unit, branch, member or subsidiary of {b}, or is subordinated to {b}'),
        'HAS_PART': ('PART_OF', 'b', 'the sentence says {b} is a unit, branch, member or subsidiary of {a}, or is subordinated to {a}'),
        'ALLIED_WITH': ('ALLIED_WITH', None, '{a} and {b} cooperated, supported, funded or allied with each other'),
        'OPPOSES': ('OPPOSES', 'a', '{a} fought, attacked, accused, sanctioned or criticized {b}'),
        'OPPOSED_BY': ('OPPOSES', 'b', '{b} fought, attacked, accused, sanctioned or criticized {a}'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
    'LOC_LOC': {
        'SAME_AS': ('SAME_AS', None, '{a} and {b} are the same place: another name, spelling or former name'),
        'PART_OF': ('PART_OF', 'a', '{a} is located inside or is part of {b}'),
        'HAS_PART': ('PART_OF', 'b', '{b} is located inside or is part of {a}'),
        'ATTACKED': ('ATTACKED', 'a', '{a} (as a state or its forces) attacked, invaded, shelled or struck {b}'),
        'ATTACKED_BY': ('ATTACKED', 'b', '{b} (as a state or its forces) attacked, invaded, shelled or struck {a}'),
        'CONTROLS': ('CONTROLS', 'a', '{a} occupies, annexed or controls {b}, or part of {b} is controlled by {a}'),
        'CONTROLLED_BY': ('CONTROLS', 'b', '{b} occupies, annexed or controls {a}, or part of {a} is controlled by {b}'),
        'SUPPORTS': ('SUPPORTS', None, 'one of {a} and {b} (as states) gave aid, weapons, funding or diplomatic support to the other'),
        'NEGOTIATED_WITH': ('NEGOTIATED_WITH', None, '{a} and {b} (as states) held talks, negotiated or signed an agreement'),
        'CO_OCCURS_WITH': ('CO_OCCURS_WITH', None, '{a} and {b} are only mentioned together, e.g. as items of a list; the sentence does not say how they are related'),
    },
}

TYPE_NAME = {'PER': 'person', 'ORG': 'organization', 'LOC': 'location'}

def classify_relation(a, b, type_a, type_b, sentence):
    """One von decision for the pair in its sentence -> (relation, subject entity or None, confidence)."""
    schema = RELATION_SCHEMAS[f'{type_a}_{type_b}']
    result = von.decide(
        state=sentence,
        choices={key: description.format(a=a, b=b) for key, (_, _, description) in schema.items()},
        instructions=f'Which relation between "{a}" ({TYPE_NAME[type_a]}) and "{b}" ({TYPE_NAME[type_b]}) does this sentence state explicitly? Choose CO_OCCURS_WITH if the sentence only mentions them together.'
    )
    relation, subject, _ = schema[result.choice]
    return relation, {'a': a, 'b': b, None: None}[subject], round(result.confidence, 2)


###Optimizing the Max_Relation_Distance

In [ ]:
# 1. Filter clean entities
clean_df = df[df['entity_group'].isin(['PER', 'LOC', 'ORG'])].copy()

pairs_data = []

# Group by paragraph
grouped = clean_df.groupby(['Source', 'Index'])

for (source, idx), group in grouped:
    entities = group[['word', 'entity_group', 'start', 'end']].drop_duplicates(subset=['word']).sort_values('start').to_dict('records')
    text = group['Text'].iloc[0]

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            e1, e2 = entities[i], entities[j]

            # Ensure e1 is the earlier entity in the text
            if e1['start'] > e2['start']:
                e1, e2 = e2, e1

            # Calculate character distance and extract text between them
            if e1['end'] <= e2['start']:
                dist = e2['start'] - e1['end']
                text_between = text[e1['end']:e2['start']]
            else:
                dist = 0
                text_between = "[OVERLAPPING]"

            # Extended Context Snippet: 40 chars before e1 -> 40 chars after e2
            start_bound = max(0, e1['start'] - 40)
            end_bound = min(len(text), e2['end'] + 40)
            context_snippet = text[start_bound:end_bound]

            pairs_data.append({
                'Source': source,
                'Paragraph Index': idx,
                'Entity 1': f"{e1['word']} ({e1['entity_group']})",
                'Entity 2': f"{e2['word']} ({e2['entity_group']})",
                'Distance (chars)': dist,
                'Text Between': text_between,
                'Context Snippet (±40 chars)': context_snippet,
                'Full Text': text
            })

            if len(pairs_data) >= 100:
                break
        if len(pairs_data) >= 100:
            break
    if len(pairs_data) >= 100:
        break

df_pairs = pd.DataFrame(pairs_data)

# Distance Binned Summary
bins = [0, 20, 40, 60, 80, 120, 200, 1000]
labels = ['0-20', '21-40', '41-60', '61-80', '81-120', '121-200', '200+']
df_pairs['Distance Bin'] = pd.cut(df_pairs['Distance (chars)'], bins=bins, labels=labels, include_lowest=True)

print(f"Sampled {len(df_pairs)} entity pairs across {df_pairs['Source'].nunique()} sources.\n")
print("=== Distance Bucket Distribution ===")
print(df_pairs['Distance Bin'].value_counts().sort_index())

In [ ]:
# Display sample rows including the new Context Snippet column
pd.set_option('display.max_colwidth', None)

# Sample up to 4 rows per distance bin
sample_inspection = df_pairs.sort_values('Distance Bin').groupby('Distance Bin', observed=False).head(4).reset_index(drop=True)

display_cols = [
    'Distance Bin',
    'Distance (chars)',
    'Entity 1',
    'Entity 2',
    'Text Between',
    'Context Snippet (±40 chars)'
]

print("=== Text & Context Inspection Between Entity Pairs ===")
sample_inspection[display_cols]

##Building the Graph - Nodes and Edges

###Text Index & Entity Pairs

In [ ]:
DECAY_LAMBDA = 0.01
MAX_RELATION_DISTANCE = 100  # 100-character distance gate
SENTENCE_END = ('.', '!', '?', '\n')
TYPE_ORDER = {'PER': 0, 'ORG': 1, 'LOC': 2}

clean_df = df[df['entity_group'].isin(['PER', 'LOC', 'ORG'])].copy()

# Text index: every unique text in the original order -> text_id 0..N-1
texts = df.drop_duplicates(['Source', 'Index'])[['Source', 'Index', 'Text', 'title', 'sub_title']].reset_index(drop=True)
text_id_of = {key: i for i, key in enumerate(zip(texts['Source'], texts['Index']))}
clean_df['text_id'] = [text_id_of[key] for key in zip(clean_df['Source'], clean_df['Index'])]

# One type per entity: the type the NER model assigned most often
entity_type = clean_df.groupby('word')['entity_group'].agg(lambda s: s.value_counts().index[0]).to_dict()

def gap(m1, m2, text):
    """Character distance d_k = max(0, start_2 - end_1) between two mentions, and the text between them."""
    (s1, e1), (s2, e2) = sorted([m1, m2])
    return (s2 - e1, text[e1:s2]) if e1 <= s2 else (0, '')

def is_comma_list(m1, m2, text, spans):
    """True when only commas separate two mentions once other entity mentions are removed (items of one list)."""
    (s1, e1), (s2, e2) = sorted([m1, m2])
    between = list(text[e1:s2])
    for s, e in spans:
        for k in range(max(s, e1), min(e, s2)):
            between[k - e1] = ' '
    return bool(re.fullmatch(r'[\s,;]*,[\s,;]*', ''.join(between)))

def sentence_of(text, start, end):
    """The sentence containing [start, end), cut at the same sentence-ending characters as the dual gate."""
    left = max(text.rfind(p, 0, start) for p in SENTENCE_END) + 1
    right = min([i for i in (text.find(p, end) for p in SENTENCE_END) if i != -1], default=len(text) - 1)
    return text[left:right + 1].strip()

# Occurrences of an entity pair in a text:
#  - one per sentence in which both entities pass the dual gate (closest mentions in that sentence) -> von.
#    von sees only the sentence and the two names, so more mention pairs in the same sentence would get the same answer
#  - if no sentence passes the gate, one CO_OCCURS_WITH occurrence (closest mentions in the text)
#  - two items of one comma list ("Horlivka, Mariupol, Yenakiieve") pass the gate but are flagged comma_list: no von
# The weight still counts one distance per text (primary): d_k of the closest mentions, preferring the gate
pair_rows = []
for (source, idx), group in tqdm(clean_df.groupby(['Source', 'Index'], sort=False), desc='Collecting entity pairs'):
    text = group['Text'].iloc[0]
    mentions = {w: list(zip(g['start'], g['end'])) for w, g in group.groupby('word', sort=False)}
    spans = list(zip(group['start'], group['end']))
    words = list(mentions)

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            best, per_sentence = None, {}
            for mu in mentions[words[i]]:
                for mv in mentions[words[j]]:
                    dist, between = gap(mu, mv, text)
                    gate = dist <= MAX_RELATION_DISTANCE and not any(p in between for p in SENTENCE_END)
                    if best is None or (not gate, dist) < (not best[3], best[2]):
                        best = (mu, mv, dist, gate)
                    if gate:
                        sentence_start = max(text.rfind(p, 0, min(mu[0], mv[0])) for p in SENTENCE_END)
                        if sentence_start not in per_sentence or dist < per_sentence[sentence_start][2]:
                            per_sentence[sentence_start] = (mu, mv, dist, gate)

            for mu, mv, dist, gate in (list(per_sentence.values()) or [best]):
                # Canonical order for the relation schemas: PER -> ORG -> LOC, same type -> first mentioned
                a, b = (words[i], mu), (words[j], mv)
                if (TYPE_ORDER[entity_type[b[0]]], b[1]) < (TYPE_ORDER[entity_type[a[0]]], a[1]):
                    a, b = b, a
                pair_rows.append((text_id_of[(source, idx)], source, a[0], b[0], int(a[1][0]), int(a[1][1]),
                                  int(b[1][0]), int(b[1][1]), dist, gate, (mu, mv) == best[:2],
                                  gate and is_comma_list(mu, mv, text, spans)))

pairs_df = pd.DataFrame(pair_rows, columns=['text_id', 'Source', 'a', 'b', 'a_start', 'a_end', 'b_start', 'b_end',
                                            'distance', 'gate', 'primary', 'comma_list'])
print(f"{len(texts)} texts, {len(entity_type)} entities, {pairs_df['primary'].sum()} entity pairs in texts, "
      f"{len(pairs_df)} occurrences, {pairs_df['gate'].sum()} pass the dual gate, "
      f"{pairs_df['comma_list'].sum()} of them comma lists -> {(pairs_df['gate'] & ~pairs_df['comma_list']).sum()} to von")


###Relation Classification (von)

In [ ]:
import hashlib

# von results are kept in the repo and reused in every run; new decisions are added to the same file
VON_RESULTS_PATH = 'Corpus2GeoRAG/Graph_n_Geocoding/von_results.parquet'
MIN_CONFIDENCE = 0.4  # weaker decisions fall back to CO_OCCURS_WITH (applied below, not in the cache)

# von decides from the sentence and the two entities only, so a result is stored by (a, b, types, sentence) and reused
# for the same pair in the same sentence: in other texts, other runs and new corpus versions.
# SCHEMA_ID changes when RELATION_SCHEMAS changes, so results of an older schema are not reused
SCHEMA_ID = hashlib.md5(repr(RELATION_SCHEMAS).encode()).hexdigest()[:8]
KEY = ['a', 'b', 'type_a', 'type_b', 'sentence']

# Pairs that pass the dual gate and are not items of one comma list, with their sentence
gated = pairs_df[pairs_df['gate'] & ~pairs_df['comma_list']].copy()
gated['type_a'], gated['type_b'] = gated['a'].map(entity_type), gated['b'].map(entity_type)
gated['sentence'] = [sentence_of(texts.at[t, 'Text'], min(s1, s2), max(e1, e2))
                     for t, s1, e1, s2, e2 in zip(gated['text_id'], gated['a_start'], gated['a_end'], gated['b_start'], gated['b_end'])]

cache = pd.read_parquet(VON_RESULTS_PATH) if os.path.exists(VON_RESULTS_PATH) else pd.DataFrame(columns=KEY + ['schema'])
von_records = cache[cache['schema'] == SCHEMA_ID].to_dict('records')
done = {tuple(r[k] for k in KEY) for r in von_records}
todo = gated.drop_duplicates(KEY)
todo = todo[[key not in done for key in zip(*(todo[k] for k in KEY))]]
print(f'{len(gated)} pairs for von, {gated.drop_duplicates(KEY).shape[0]} unique (pair, sentence): '
      f'{len(von_records)} already classified, {len(todo)} to classify')

# Classify; the results file is saved every 2000 decisions and at the end
for n, row in enumerate(tqdm(todo.itertuples(index=False), total=len(todo), desc='Classifying relations (von)'), 1):
    try:
        relation, subject, confidence = classify_relation(row.a, row.b, row.type_a, row.type_b, row.sentence)
        von_records.append({'a': row.a, 'b': row.b, 'type_a': row.type_a, 'type_b': row.type_b, 'sentence': row.sentence,
                            'relation': relation, 'subject': subject, 'confidence': confidence, 'schema': SCHEMA_ID})
    except Exception as e:
        tqdm.write(f'Skipped {row.a!r} / {row.b!r} in text {row.text_id} ({type(e).__name__}), will retry on next run')
    if n % 2000 == 0 or n == len(todo):
        pd.DataFrame(von_records).to_parquet(VON_RESULTS_PATH, index=False)

# Attach the results: pairs that failed the gate, and decisions below MIN_CONFIDENCE, are CO_OCCURS_WITH
von_df = pd.DataFrame(von_records, columns=KEY + ['relation', 'subject', 'confidence'])
gated = gated.merge(von_df, on=KEY, how='left')
OCCURRENCE = ['text_id', 'a', 'b', 'a_start', 'b_start']
pairs_df = pairs_df.drop(columns=['relation', 'subject', 'confidence'], errors='ignore').merge(
    gated[OCCURRENCE + ['relation', 'subject', 'confidence']], on=OCCURRENCE, how='left')
weak = pairs_df['confidence'] < MIN_CONFIDENCE
pairs_df.loc[weak, 'relation'] = 'CO_OCCURS_WITH'
pairs_df.loc[weak, 'subject'] = None
pairs_df['relation'] = pairs_df['relation'].fillna('CO_OCCURS_WITH')
pairs_df[['subject', 'confidence']] = pairs_df[['subject', 'confidence']].astype(object).where(pairs_df[['subject', 'confidence']].notna(), None)

print(f"\n{weak.sum()} decisions below MIN_CONFIDENCE = {MIN_CONFIDENCE} set to CO_OCCURS_WITH\n")
print(pairs_df.loc[pairs_df['gate'], 'relation'].value_counts())


###Graph Assembly

In [ ]:
G = nx.Graph()

# Entity nodes; LOC nodes carry the geocoding fields
geo_attrs = geo_df.set_index('word')[GEO_COLS]
for entity, etype in entity_type.items():
    attrs = {'type': etype}
    if etype == 'LOC':
        values = geo_attrs.loc[entity] if entity in geo_attrs.index else {}
        attrs.update({col: (None if pd.isna(values.get(col)) else values.get(col)) for col in GEO_COLS})
    G.add_node(entity, **attrs)

# Text nodes, in the original order
for text_id, (source, index, text, title, sub_title) in enumerate(
        texts[['Source', 'Index', 'Text', 'title', 'sub_title']].itertuples(index=False, name=None)):
    G.add_node(f'TEXT_{text_id}', type='TEXT', text_id=text_id, text=text, source=source, index=int(index),
               title=None if pd.isna(title) else title, sub_title=None if pd.isna(sub_title) else sub_title)

# Entity-entity edges: one record per occurrence (pair in a sentence, or in a text when no sentence passes the gate).
# NetworkX exports an undirected edge with the earlier-added node as 'source', so spans are oriented the same way
node_order = {n: i for i, n in enumerate(G.nodes)}
for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df), desc='Building entity edges'):
    u, v, u_span, v_span = row.a, row.b, (row.a_start, row.a_end), (row.b_start, row.b_end)
    if node_order[u] > node_order[v]:
        u, v, u_span, v_span = v, u, v_span, u_span

    if not G.has_edge(u, v):
        G.add_edge(u, v, c_eff=0.0, sources=set(), relations={})
    data = G[u][v]
    if row.primary:  # C_eff counts one distance per text
        data['c_eff'] += math.exp(-DECAY_LAMBDA * row.distance)
    data['sources'].add(row.Source)
    data['relations'].setdefault(row.relation, []).append({
        'text_id': row.text_id, 'source_start': u_span[0], 'source_end': u_span[1],
        'target_start': v_span[0], 'target_end': v_span[1], 'subject': row.subject, 'confidence': row.confidence})

# W = max(0.1, round(S * ln(1 + C_eff), 2)); only weight and relations stay on the edge
for u, v, data in G.edges(data=True):
    data['weight'] = max(0.1, round(len(data.pop('sources')) * math.log1p(data.pop('c_eff')), 2))
n_entity_edges = G.number_of_edges()

# IN_THE_TEXT edges: entity -> text node, with every mention of the entity in that text
mentions_df = clean_df.drop_duplicates(['word', 'text_id', 'start'])
for (word, text_id), g in tqdm(mentions_df.groupby(['word', 'text_id'], sort=False), desc='Building IN_THE_TEXT edges'):
    G.add_edge(word, f'TEXT_{text_id}', weight=len(g),
               relations={'IN_THE_TEXT': [{'start': int(s), 'end': int(e)} for s, e in zip(g['start'], g['end'])]})

print(f"\nGraph created: {G.number_of_nodes()} nodes ({len(entity_type)} entities, {len(texts)} texts), "
      f"{n_entity_edges} entity edges, {G.number_of_edges() - n_entity_edges} IN_THE_TEXT edges.")


###Spatial Relations (INSIDE)

In [ ]:
# INSIDE: location A is inside location B when A's point lies in B's polygon.
# Only the smallest containing polygon is linked (direct parent), using the ~50 m geometry_fine to stay exact at borders
spatial = geo_df[geo_df['word'].map(entity_type) == 'LOC'].dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
geoms = [shapely.from_wkt(w) if isinstance(w, str) else None for w in spatial['geometry_fine']]
areas = np.array([g.area if g is not None else 0.0 for g in geoms])
area_idx = np.array([i for i, g in enumerate(geoms) if g is not None and g.geom_type in ('Polygon', 'MultiPolygon')])

tree = shapely.STRtree([geoms[i] for i in area_idx])
inner_idx, tree_idx = tree.query(shapely.points(spatial['longitude'], spatial['latitude']), predicate='within')

parent = {}
for i, j in zip(inner_idx, area_idx[tree_idx]):
    if i != j and areas[j] > areas[i] and (i not in parent or areas[j] < areas[parent[i]]):
        parent[i] = j

for i, j in parent.items():
    inner, outer = spatial.at[i, 'word'], spatial.at[j, 'word']
    record = {'subject': inner, 'basis': 'geometry'}
    if G.has_edge(inner, outer):
        G[inner][outer]['relations'].setdefault('INSIDE', []).append(record)
    else:
        G.add_edge(inner, outer, weight=0.1, relations={'INSIDE': [record]})  # no text: S = 0 -> W = max(0.1, 0) = 0.1

print(f'{len(parent)} INSIDE relations ({len(spatial)} located LOC entities, {len(area_idx)} with a polygon)')
pd.DataFrame([(spatial.at[i, 'word'], spatial.at[j, 'word']) for i, j in list(parent.items())[:10]], columns=['inside', 'parent'])


###Relation Fixes
von alone is right about half of the time on relations other than `CO_OCCURS_WITH`, and its confidence does not tell the right ones from the wrong ones. These fixes run on the built graph, so every relation gets them, new ones included. Run them once per built graph.

1. **Junk names:** names that are NER mistakes (`.`, `Armed Forces of`, `Army` alone) leave the graph. Names made only of numbers stay.
2. **Lists:** `A, B and C` become `CO_OCCURS_WITH`.
3. **Landmarks and sources:** `X, south of Y` between two places, and `according to X`, become `CO_OCCURS_WITH`.
4. **Patterns von missed:** `Place, Region` inside our polygons becomes `PART_OF`, `mayor of X, Name` becomes `LEADS`, `Long Name (LN)` or the same name in another script becomes `SAME_AS`, `was born / was killed in` becomes `BORN_IN` / `DIED_IN`, `Unit (Place)` becomes `BASED_IN`.
5. **Second model:** a text-checking model reads each remaining relation with its sentence. A relation below `MIN_CHECK` becomes `CO_OCCURS_WITH`.

Every record a fix changes gets `confidence` 0.77. von's own scores can also be 0.77, so `relation_fixes.parquet` is what tells them apart: it lists each change with the name of the fix. It also keeps the second model's scores, so later runs only score new relations (same logic as `von_results.parquet`).

In [ ]:
# Relation fixes, part 1: rules on the built graph. von picks a relation for the whole sentence, so it often answers about
# other entities in it; these rules go by what the text shows between the two names. Run once per built graph.
from collections import Counter, defaultdict, namedtuple

CO = 'CO_OCCURS_WITH'
FIXED_CONFIDENCE = 0.77  # confidence of every record a fix changes; relation_fixes.parquet tells them from von's own 0.77

# Mention positions of every text and mention counts of every entity, from the IN_THE_TEXT edges
entity_nodes = {n: t for n, t in G.nodes(data='type') if t != 'TEXT'}
spans_of, mention_count = defaultdict(set), Counter()
for u, v, data in G.edges(data=True):
    if 'IN_THE_TEXT' in data['relations']:
        word, text_node = (u, v) if G.nodes[v]['type'] == 'TEXT' else (v, u)
        spans_of[G.nodes[text_node]['text_id']].update((r['start'], r['end']) for r in data['relations']['IN_THE_TEXT'])
        mention_count[word] += len(data['relations']['IN_THE_TEXT'])
spans_of = {t: sorted(spans) for t, spans in spans_of.items()}

# Fix 1: names that are NER mistakes (punctuation, a name cut in the middle, a generic word alone) leave the graph.
# Names made only of numbers stay: "9th" or "1514" can be real units, so they are left to the NER team
FUNCTION_WORD = r'(?:of|the|and|for|in|on|at|to|by|with|from)'
GENERIC_NAMES = {'Brigade', 'Battalion', 'Regiment', 'Division', 'Company', 'Squadron', 'Corps', 'Platoon', 'Detachment', 'Defense', 'Defence',
                 'Army', 'Navy', 'Ministry', 'Council', 'Republic', 'Guards', 'Russian', 'Ukrainian', 'Soviet', 'American', 'Black', 'South',
                 'North', 'East', 'Government', 'Parliament', 'Forces', 'Military', 'Command', 'Service', 'Office', 'Unit', 'Group', 'Agency',
                 'Police', 'Court', 'Board', 'Mission', 'Lake', 'School', 'Order', 'Marines', 'The'}
NEXT_SENTENCE_WORDS = {'In', 'Two', 'During', 'Following', 'According', 'Shortly', 'After', 'This', 'Fortunately', 'Special', 'To', 'The', 'On', 'At'}
KEEP_LOWERCASE = re.compile(r'^(?:al-|el-|von |van |de |la |le |i[A-Z]|open[A-Z]|root\d)|\.(?:com|org|net|ua|ru|in\.ua)$')

def junk_reason(name):
    """Why an entity name is a NER mistake, or None."""
    s = name.strip()
    if not any(c.isalnum() for c in s): return 'punctuation only'
    if len(s) == 1 and s.isalpha() and s != 'X': return 'single letter'
    if re.search(rf'\s{FUNCTION_WORD}$', s, re.I) or s.lower() in ('of', 'the', 'and', 'of the'): return 'ends in the middle of a name'
    if re.match(rf'^{FUNCTION_WORD}\s', s): return 'starts in the middle of a name'
    joined = re.search(r'[a-z]\.([A-Z][a-z]+)$', s)
    if joined and joined.group(1) in NEXT_SENTENCE_WORDS: return 'joined with the next sentence'
    if s in GENERIC_NAMES: return 'generic word'
    if s[0].islower() and s[0].isascii() and ' ' not in s and '-' not in s and not KEEP_LOWERCASE.search(s):
        twin = s[0].upper() + s[1:]   # "crimea" is only a spelling of a known entity
        return None if entity_nodes.get(twin) == entity_nodes[name] and mention_count[twin] >= 50 else 'lowercase common word'
    return None

junk_names = {w: junk_reason(w) for w in entity_nodes if junk_reason(w)}

# A place whose direct parent is a junk name moves to the next smallest polygon that contains it (same logic as INSIDE)
spatial_words = spatial['word'].tolist()
next_parent = {}
for i, j in zip(inner_idx, area_idx[tree_idx]):
    if i != j and areas[j] > areas[i] and spatial_words[j] not in junk_names and (i not in next_parent or areas[j] < areas[next_parent[i]]):
        next_parent[i] = j
next_parent = {spatial_words[i]: spatial_words[j] for i, j in next_parent.items()}
relinked = [(other, next_parent[other]) for name in junk_names for other, data in G[name].items()
            if other not in junk_names and other in next_parent and any(r['subject'] == other for r in data['relations'].get('INSIDE', []))]
for inner, outer in relinked:
    record = {'subject': inner, 'basis': 'geometry'}
    if G.has_edge(inner, outer):
        G[inner][outer]['relations'].setdefault('INSIDE', []).append(record)
    else:
        G.add_edge(inner, outer, weight=0.1, relations={'INSIDE': [record]})
G.remove_nodes_from(junk_names)
entity_type = {w: t for w, t in entity_type.items() if w not in junk_names}

# Fix 2: "A, B and C" are items of a list. Two names can still act together ("A and B met"), so joint relations need 3+ items
LIST_SEPARATOR = re.compile(r'[\s,;/&]*(?:,|;|/|&|\band\b|\bor\b|\bas well as\b)[\s,;/&]*(?:\band\b|\bor\b)?[\s,;/&]*', re.I)
JOINT_RELATIONS = {'MET_WITH', 'NEGOTIATED_WITH', 'ALLIED_WITH', 'RELATIVE_OF'}

def is_list(p, relation):
    if not LIST_SEPARATOR.fullmatch(p['between']) or re.search(r'\bbetween\s+(?:the\s+)?$', p['before'], re.I):
        return False
    return relation not in JOINT_RELATIONS or p['items'] >= 3

# Fix 3: "X, south of Y" uses Y as a landmark between two places; "according to X" makes X the source, not the actor
LANDMARK = re.compile(r"(?:(?:north|south|east|west)(?:[- ]?(?:north|south|east|west))*(?:ern)?\s+(?:of|from)|\bnear|\boutside(?:\s+of)?|\baround|\bclose\s+to"
                      r"|\bnot\s+far\s+from|\bnearby|\b(?:km|kilomet(?:er|re)s?|miles?)\s+(?:\S+\s+)?(?:from|of))\s+(?:the\s+)?(?:(?:city|town|village|settlement)\s+of\s+)?$", re.I)

def is_landmark(p): return p['type_a'] == 'LOC' and p['type_b'] == 'LOC' and bool(LANDMARK.search(p['between']))
def is_source(p): return bool(re.search(r'\baccording\s+to\s+(?:the\s+)?$', p['before'], re.I))

# Fix 4: patterns that state a relation von missed
TITLE = r'(?:mayor|governor|president|chairman|chairwoman|head|chief|commander|director|minister|leader|secretary)'
LIFE_EVENT = re.compile(r'\b(?:was|were)\s+(?:\w+\s+){0,2}?(born)\b|\b(native)\s+of\b|\b(?:was|were)\s+(?:\w+\s+){0,2}?(killed|buried|assassinated|executed)\b|\b(died)\b|\bfatally\s+(shot)\b', re.I)

def leads_by_title(p):
    """'mayor of X, Name' or 'Name, commander of X'."""
    if p['type_first'] != 'PER' and p['type_second'] == 'PER' and len(p['between']) < 30 and re.search(rf'\b{TITLE}\s+of\s+(?:the\s+)?(?:\w+\s+)?$', p['before'], re.I):
        return bool(re.fullmatch(r"[\s,]*(?:(?:\w+\s+){0,3})?", p['between']))
    return p['type_first'] == 'PER' and p['type_second'] != 'PER' and bool(re.fullmatch(rf"\s*,\s*(?:(?:the|former|acting|deputy)\s+)*{TITLE}\s+of\s+(?:the\s+)?(?:\w+\s+)?", p['between'], re.I))

def life_event(p):
    """'Name was born / was killed ... in Place'."""
    if p['type_first'] != 'PER' or p['type_second'] != 'LOC': return None
    event = LIFE_EVENT.search(p['between'])
    return None if not event else 'BORN_IN' if event.group(1) or event.group(2) else 'DIED_IN'

def initials(s): return ''.join(w[0] for w in re.findall(r"[^\W\d_]+", s) if w.lower() not in ('of', 'the', 'and', 'for', 'in', 'on', 'de')).upper()
def is_cyrillic(s): return sum('Ѐ' <= c <= 'ӿ' for c in s) > sum(c.isascii() and c.isalpha() for c in s)

def same_name(p):
    """'Long Name (LN)', or the same name in another script next to it."""
    if p['type_a'] != p['type_b'] and {p['type_a'], p['type_b']} != {'ORG', 'LOC'}: return False
    first, second = p['first'], p['second']
    if not re.fullmatch(r"\s*(?:\(|,\s*romanized:|\(\s*(?:Ukrainian|Russian)\s*:)\s*", p['between'], re.I): return False
    return (is_cyrillic(first) != is_cyrillic(second) or (initials(first) == second.replace('.', '').upper() and len(second) >= 2)
            or (initials(second) == first.replace('.', '').upper() and len(first) >= 2))

# "Place, Region": a comma is not a list when our polygons say one place is inside the other
located = geo_df.dropna(subset=['latitude', 'longitude']).drop_duplicates('word').set_index('word')
location_polygons = {}

def is_inside(inner, outer):
    if outer not in location_polygons:
        wkt = located.at[outer, 'geometry_fine'] if outer in located.index else None
        shape = shapely.from_wkt(wkt) if isinstance(wkt, str) else None
        location_polygons[outer] = shape if shape is not None and shape.geom_type in ('Polygon', 'MultiPolygon') else None
    if inner not in located.index or location_polygons[outer] is None:
        return False
    return location_polygons[outer].contains(shapely.Point(located.at[inner, 'longitude'], located.at[inner, 'latitude']))

def context_of(o):
    """The text around a pair: names in reading order, the words between them (other entity names blanked), the words
    before the first name, and how many items their list has."""
    text, spans = G.nodes[f'TEXT_{o.text_id}']['text'], spans_of[o.text_id]
    if o.a_start <= o.b_start: s1, e1, s2, e2, first, second, type_first, type_second = o.a_start, o.a_end, o.b_start, o.b_end, o.a, o.b, o.type_a, o.type_b
    else: s1, e1, s2, e2, first, second, type_first, type_second = o.b_start, o.b_end, o.a_start, o.a_end, o.b, o.a, o.type_b, o.type_a
    between = list(text[e1:s2]) if e1 <= s2 else []
    for s, e in spans:
        for k in range(max(s, e1), min(e, s2)):
            between[k - e1] = ' '
    items, position = 2 + sum(1 for s, e in spans if s >= e1 and e <= s2), s1
    for s, e in reversed([x for x in spans if x[1] <= s1]):
        if not LIST_SEPARATOR.fullmatch(text[e:position] or '-'): break
        items, position = items + 1, s
    position = e2
    for s, e in [x for x in spans if x[0] >= e2]:
        if not LIST_SEPARATOR.fullmatch(text[position:s] or '-'): break
        items, position = items + 1, e
    return dict(type_a=o.type_a, type_b=o.type_b, first=first, second=second, type_first=type_first, type_second=type_second,
                between=''.join(between), before=text[max(0, s1 - 60):s1], items=items)

def fixed_relation(o):
    """Fixes 2-4 for one record: (relation, subject, name of the fix), or None when nothing changes."""
    p, relation = context_of(o), o.relation
    if relation != 'SAME_AS' and same_name(p):
        return 'SAME_AS', None, 'same_name'
    if relation == CO:
        if o.type_a == 'LOC' and o.type_b == 'LOC' and re.fullmatch(r'[\s,]*,[\s,]*|\s*\(\s*', p['between']):
            if is_inside(o.a, o.b): return 'PART_OF', o.a, 'address'
            if is_inside(o.b, o.a): return 'PART_OF', o.b, 'address'
        if o.type_a == 'ORG' and o.type_b == 'LOC' and re.fullmatch(r'\s*\(\s*', p['between']) and o.a_start < o.b_start:
            return 'BASED_IN', o.a, 'unit_place'
        if o.type_a == 'PER' and leads_by_title(p):
            return 'LEADS', o.a, 'title'
    if relation not in ('BORN_IN', 'DIED_IN') and o.type_a == 'PER' and o.type_b == 'LOC' and life_event(p):
        return life_event(p), o.a, 'life_event'
    if relation not in (CO, 'STATED_ABOUT') and is_source(p):
        return CO, None, 'according_to'
    if relation != CO and is_list(p, relation):
        return CO, None, 'list'
    if relation not in (CO, 'SAME_AS') and is_landmark(p):
        return CO, None, 'landmark'
    return None

# Every text record of two names in one sentence, in the order von saw them (PER -> ORG -> LOC, same type -> first mentioned).
# Pairs that only share a text have no sentence to read, so they are left alone
Occurrence = namedtuple('Occurrence', 'u v record relation subject text_id a b a_start a_end b_start b_end type_a type_b sentence')
text_relations = []
for u, v, data in G.edges(data=True):
    if 'IN_THE_TEXT' in data['relations']:
        continue
    for relation, records in data['relations'].items():
        if relation == 'INSIDE':
            continue
        for record in records:
            text = G.nodes[f"TEXT_{record['text_id']}"]['text']
            spans = [(u, record['source_start'], record['source_end']), (v, record['target_start'], record['target_end'])]
            if text[spans[0][1]:spans[0][2]] != u:   # after loading a graph from its file the two ends can be swapped
                spans = [(v, record['source_start'], record['source_end']), (u, record['target_start'], record['target_end'])]
            (a, a_start, a_end), (b, b_start, b_end) = sorted(spans, key=lambda x: (TYPE_ORDER[entity_nodes[x[0]]], x[1], x[2]))
            distance, between = gap((a_start, a_end), (b_start, b_end), text)
            if distance <= MAX_RELATION_DISTANCE and not any(p in between for p in SENTENCE_END):
                text_relations.append(Occurrence(u, v, record, relation, record['subject'], record['text_id'], a, b, a_start, a_end, b_start, b_end,
                                                 entity_nodes[a], entity_nodes[b], sentence_of(text, min(a_start, b_start), max(a_end, b_end))))

rule_changes = [fixed_relation(o) for o in tqdm(text_relations, desc='Relation fixes (rules)')]
print(f"{len(junk_names)} junk names removed: {pd.Series(junk_names).value_counts().to_dict()}")
print(f"{len(relinked)} INSIDE links moved from a junk name to the next parent")
print(f"Records changed by the rules: {Counter(change[2] for change in rule_changes if change).most_common()}")

In [ ]:
# Relation fixes, part 2: a second model checks each remaining von relation against its sentence. It is trained for one
# question: does this text state this claim? Each relation is written as short one-verb claims and the best score counts
CHECK_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
MIN_CHECK = 0.5  # below this, the sentence does not state the relation

RELATION_CLAIMS = {
    'PER_LOC': {
        'BORN_IN': ['{a} was born in {b}', '{a} comes from {b}', '{a} is a native of {b}'],
        'LOCATED_IN': ['{a} lived in {b}', '{a} worked in {b}', '{a} served in {b}', '{a} fought in {b}', '{a} was held in {b}', '{a} was in {b}'],
        'VISITED': ['{a} visited {b}', '{a} travelled to {b}', '{a} arrived in {b}'],
        'FLED_TO': ['{a} fled to {b}', '{a} emigrated to {b}', '{a} was deported to {b}', '{a} defected to {b}'],
        'DIED_IN': ['{a} died in {b}', '{a} was killed in {b}', '{a} was buried in {b}'],
        'LEADS': ['{a} is the president of {b}', '{a} is the prime minister of {b}', '{a} is a minister of {b}', '{a} is the governor of {b}', '{a} is the mayor of {b}', '{a} is the head of {b}'],
        'OPPOSES': ['{a} attacked {b}', '{a} threatened {b}', '{a} acted against {b}'],
        'OPPOSED_BY': ['{b} attacked {a}', '{b} killed {a}', '{b} sanctioned {a}', '{b} arrested {a}', '{b} banned {a}', '{b} accused {a}'],
        'STATED_ABOUT': ['{a} said something about {b}', '{a} made a statement about {b}'],
    },
    'PER_ORG': {
        'MEMBER_OF': ['{a} is a member of {b}', '{a} works for {b}', '{a} is an official of {b}', '{a} served in {b}'],
        'LEADS': ['{a} leads {b}', '{a} commands {b}', '{a} is the head of {b}', '{a} founded {b}'],
        'OPPOSES': ['{a} opposed {b}', '{a} criticized {b}', '{a} attacked {b}', '{a} fought against {b}'],
        'OPPOSED_BY': ['{b} accused {a}', '{b} criticized {a}', '{b} sanctioned {a}', '{b} arrested {a}', '{b} detained {a}', '{b} attacked {a}', '{b} killed {a}'],
        'STATED_ABOUT': ['{b} reported something about {a}', '{a} said something about {b}'],
    },
    'PER_PER': {
        'SAME_AS': ['{a} and {b} are the same person', '{b} is another name of {a}'],
        'RELATIVE_OF': ['{a} and {b} are family', '{a} is married to {b}', '{a} is a relative of {b}'],
        'MET_WITH': ['{a} met {b}', '{a} talked with {b}', '{a} negotiated with {b}'],
        'ALLIED_WITH': ['{a} and {b} are allies', '{a} and {b} are colleagues', '{a} worked with {b}', '{a} supported {b}'],
        'OPPOSES': ['{a} opposed {b}', '{a} criticized {b}', '{a} attacked {b}', '{a} accused {b}'],
        'OPPOSED_BY': ['{b} opposed {a}', '{b} criticized {a}', '{b} attacked {a}', '{b} accused {a}'],
    },
    'ORG_LOC': {
        'BASED_IN': ['{a} is based in {b}', '{a} is headquartered in {b}', '{a} is stationed in {b}'],
        'OPERATES_IN': ['{a} operates in {b}', '{a} fought in {b}', '{a} was deployed in {b}'],
        'ATTACKED': ['{a} attacked {b}', '{a} shelled {b}', '{a} bombed {b}', '{a} invaded {b}', '{a} struck {b}'],
        'CONTROLS': ['{a} controls {b}', '{a} captured {b}', '{a} occupied {b}', '{a} governs {b}'],
        'SUPPORTS': ['{a} supports {b}', '{a} gave aid to {b}', '{a} gave weapons to {b}', '{b} supports {a}', '{b} gave aid to {a}', '{b} gave weapons to {a}'],
        'OPPOSES': ['{a} sanctioned {b}', '{a} accused {b}', '{a} acted against {b}', '{b} sanctioned {a}', '{b} accused {a}', '{b} acted against {a}'],
    },
    'ORG_ORG': {
        'SAME_AS': ['{a} and {b} are the same organization', '{b} is another name of {a}'],
        'PART_OF': ['{a} is part of {b}', '{a} is a unit of {b}', '{a} is subordinated to {b}'],
        'HAS_PART': ['{b} is part of {a}', '{b} is a unit of {a}', '{b} is subordinated to {a}'],
        'ALLIED_WITH': ['{a} cooperated with {b}', '{a} supported {b}', '{a} is allied with {b}'],
        'OPPOSES': ['{a} fought {b}', '{a} attacked {b}', '{a} accused {b}', '{a} sanctioned {b}', '{a} criticized {b}'],
        'OPPOSED_BY': ['{b} fought {a}', '{b} attacked {a}', '{b} accused {a}', '{b} sanctioned {a}', '{b} criticized {a}'],
    },
    'LOC_LOC': {
        'SAME_AS': ['{a} and {b} are the same place', '{b} is another name of {a}'],
        'PART_OF': ['{a} is part of {b}', '{a} is located in {b}'],
        'HAS_PART': ['{b} is part of {a}', '{b} is located in {a}'],
        'ATTACKED': ['{a} attacked {b}', '{a} invaded {b}', '{a} shelled {b}', '{a} struck {b}'],
        'ATTACKED_BY': ['{b} attacked {a}', '{b} invaded {a}', '{b} shelled {a}', '{b} struck {a}'],
        'CONTROLS': ['{a} controls {b}', '{a} occupied {b}', '{a} annexed {b}'],
        'CONTROLLED_BY': ['{b} controls {a}', '{b} occupied {a}', '{b} annexed {a}'],
        'SUPPORTS': ['{a} supports {b}', '{a} gave aid to {b}', '{a} gave weapons to {b}', '{b} supports {a}', '{b} gave aid to {a}', '{b} gave weapons to {a}'],
        'NEGOTIATED_WITH': ['{a} negotiated with {b}', '{a} and {b} held talks', '{a} and {b} signed an agreement'],
    },
}

# The scores and every change are kept in the repo like von's results; later runs only score relations that are new.
# CHECK_ID changes when the model or the claims change, so scores of an older version are not reused
FIXES_PATH = 'Corpus2GeoRAG/Graph_n_Geocoding/relation_fixes.parquet'
CHECK_ID = hashlib.md5(repr((CHECK_MODEL, RELATION_CLAIMS)).encode()).hexdigest()[:8]
FIX_KEY = KEY + ['relation', 'subject']   # subject '' = symmetric relation

def fix_key(o): return (o.a, o.b, o.type_a, o.type_b, o.sentence, o.relation, o.subject or '')

def claims_of(a, b, type_a, type_b, sentence, relation, subject):
    """The short claims of a von decision: the schema choice it picked, with the real names filled in."""
    role = 'a' if subject == a else 'b' if subject == b else None
    choice = next(key for key, (rel, sub, _) in RELATION_SCHEMAS[f'{type_a}_{type_b}'].items() if rel == relation and sub == role)
    return [claim.format(a=a, b=b) + '.' for claim in RELATION_CLAIMS[f'{type_a}_{type_b}'][choice]]

def entailment_scores(premises, claims):
    """How strongly each sentence states its claim (0-1). Similar lengths go together: much faster on a GPU."""
    order, scores, i = sorted(range(len(claims)), key=lambda k: len(premises[k]) + len(claims[k])), [0.0] * len(claims), 0
    while i < len(order):
        length = len(premises[order[i]]) + len(claims[order[i]])
        size = 256 if length < 200 else 128 if length < 400 else 64 if length < 800 else 24
        while True:
            batch = order[i:i + size]
            try:
                inputs = check_tokenizer([premises[k] for k in batch], [claims[k] for k in batch], truncation=True, max_length=256,
                                         padding=True, return_tensors='pt').to(check_device)
                with torch.no_grad():
                    probabilities = torch.softmax(check_model(**inputs).logits.float(), -1)[:, check_entailment].tolist()
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                size = max(1, size // 2)
        for k, probability in zip(batch, probabilities):
            scores[k] = probability
        i += len(batch)
    return scores

def new_fix_row(key): return {**dict(zip(FIX_KEY, key)), 'check': None, 'check_id': CHECK_ID, 'fix': None, 'new_relation': None, 'new_subject': None}
def save_fixes(): pd.DataFrame(list(fix_rows.values())).to_parquet(FIXES_PATH, index=False)

# Scores already in the repo; a fix name is decided again in every run because the rules take only seconds
fix_rows = {}
if os.path.exists(FIXES_PATH):
    saved_fixes = pd.read_parquet(FIXES_PATH)
    saved_fixes = saved_fixes[(saved_fixes['check_id'] == CHECK_ID) & saved_fixes['check'].notna()]
    for row in saved_fixes[FIX_KEY + ['check']].to_dict('records'):
        fix_rows[tuple(row[k] for k in FIX_KEY)] = {**new_fix_row(tuple(row[k] for k in FIX_KEY)), 'check': row['check']}

# Only relations no rule has changed go to the second model; each decision is scored once, in length order.
# The results file is saved every 5000 decisions and at the end, and the model loads only when something is new
to_check = list(dict.fromkeys(fix_key(o) for o, change in zip(text_relations, rule_changes) if change is None and o.relation != CO))
todo_checks = sorted([key for key in to_check if key not in fix_rows], key=lambda key: len(key[4]))
print(f'{len(to_check)} relations to check: {len(to_check) - len(todo_checks)} already scored, {len(todo_checks)} to score')

if todo_checks:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    check_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    check_tokenizer = AutoTokenizer.from_pretrained(CHECK_MODEL)
    check_model = AutoModelForSequenceClassification.from_pretrained(CHECK_MODEL).to(check_device).eval()
    if check_device == 'cuda':
        check_model = check_model.half()
    check_entailment = next(i for i, label in check_model.config.id2label.items() if label.lower().startswith('entail'))

    for start in tqdm(range(0, len(todo_checks), 5000), desc='Checking relations (second model)'):
        chunk = todo_checks[start:start + 5000]
        claims = [claims_of(*key) for key in chunk]
        flat = entailment_scores([key[4] for key, c in zip(chunk, claims) for _ in c], [claim for c in claims for claim in c])
        position = 0
        for key, c in zip(chunk, claims):
            fix_rows[key] = {**new_fix_row(key), 'check': round(max(flat[position:position + len(c)]), 4)}
            position += len(c)
        save_fixes()

# Move every changed record to its new relation, with FIXED_CONFIDENCE; relations that pass the check keep von's score.
# A relation below MIN_CHECK is not stated by its sentence and falls back to CO_OCCURS_WITH
moved, touched = {}, set()
final_relations = []
for o, change in zip(text_relations, rule_changes):
    key = fix_key(o)
    if change is None and o.relation != CO and fix_rows[key]['check'] < MIN_CHECK:
        change = (CO, None, 'check')
    final_relations.append(change)
    if change:
        moved[id(o.record)] = change
        touched.add((o.u, o.v))
        row = fix_rows.setdefault(key, new_fix_row(key))
        if row['fix'] is None:
            row['fix'], row['new_relation'], row['new_subject'] = change[2], change[0], change[1] or ''

for u, v in touched:
    relations = {}
    for relation, records in G[u][v]['relations'].items():
        for record in records:
            if id(record) in moved:
                new_relation, new_subject, _ = moved[id(record)]
                record['subject'], record['confidence'] = new_subject, FIXED_CONFIDENCE
                relations.setdefault(new_relation, []).append(record)
            else:
                relations.setdefault(relation, []).append(record)
    G[u][v]['relations'] = relations
save_fixes()

before = Counter(o.relation for o in text_relations)
after = Counter(change[0] if change else o.relation for o, change in zip(text_relations, final_relations))
print(f"\n{sum(1 for change in final_relations if change and change[2] == 'check')} records not stated by their sentence set to {CO} (check < {MIN_CHECK})\n")
print(pd.DataFrame({'before fixes': before, 'after fixes': after}).fillna(0).astype(int).sort_values('after fixes', ascending=False))

###Graph Inspection

In [ ]:
records = []

# One row per relation occurrence on entity-entity edges (IN_THE_TEXT edges are counted separately)
for u, v, data in G.edges(data=True):
    for rel, occurrences in data['relations'].items():
        if rel == 'IN_THE_TEXT':
            continue
        for occ in occurrences:
            records.append({'Entity 1': u, 'Entity 2': v, 'Relation': rel, 'Subject': occ['subject'],
                            'Confidence': occ.get('confidence'), 'Weight (W)': data['weight'], 'text_id': occ.get('text_id'),
                            'start': min(occ.get('source_start', 0), occ.get('target_start', 0)),
                            'end': max(occ.get('source_end', 0), occ.get('target_end', 0))})

df_relations = pd.DataFrame(records)

# 1. Relation distribution in the graph
print("=== Relation Distribution (occurrences) ===")
print(df_relations['Relation'].value_counts())
print(f"\nIN_THE_TEXT: {sum(len(d['relations']['IN_THE_TEXT']) for _, _, d in G.edges(data=True) if 'IN_THE_TEXT' in d['relations'])} mentions")
print("\n" + "="*50 + "\n")

# 2. Highest-confidence example of each text-based semantic relation, with its sentence
semantic = df_relations[~df_relations['Relation'].isin(['CO_OCCURS_WITH', 'INSIDE'])]
examples = semantic.sort_values('Confidence', ascending=False).groupby('Relation').head(3).sort_values(['Relation', 'Confidence'], ascending=[True, False])
examples['Sentence'] = [sentence_of(texts.at[t, 'Text'], s, e) for t, s, e in zip(examples['text_id'], examples['start'], examples['end'])]

pd.set_option('display.max_colwidth', None)
examples[['Relation', 'Entity 1', 'Entity 2', 'Subject', 'Confidence', 'Weight (W)', 'Sentence']].reset_index(drop=True)


In [ ]:
# Extract entity-entity edge attributes into a list (text nodes excluded)
edge_records = []
for u, v, data in G.edges(data=True):
    if 'IN_THE_TEXT' in data['relations']:
        continue
    edge_records.append({
        'Entity 1': u,
        'Entity 2': v,
        'Weight (W)': data['weight'],
        'Occurrences': sum(len(occ) for rel, occ in data['relations'].items() if rel != 'INSIDE'),
        'Relations': ', '.join(f'{rel} ({len(occ)})' for rel, occ in data['relations'].items())
    })

# Convert to DataFrame and sort by Weight
edges_df = pd.DataFrame(edge_records).sort_values(by='Weight (W)', ascending=False).reset_index(drop=True)

# Extract Top 5, Middle 5, and Lowest 5
n = len(edges_df)
top5 = edges_df.head(5).copy()
top5.insert(0, 'Category', 'Highest 5')

mid_idx = n // 2
mid5 = edges_df.iloc[max(0, mid_idx - 2) : min(n, mid_idx + 3)].head(5).copy()
mid5.insert(0, 'Category', 'Middle 5')

bot5 = edges_df.tail(5).copy()
bot5.insert(0, 'Category', 'Lowest 5')

# Combine into a single table
summary_table = pd.concat([top5, mid5, bot5], ignore_index=True)

print(f"Total Entity Edges Analyzed: {n}\n")
summary_table


### Proximity-Aware Composite Edge Weight Equation

To quantify the connection strength between any two entities ($u, v$) in the Knowledge Graph, we use a composite weight formula that integrates **character proximity**, **co-occurrence frequency**, and **source diversity**:

$$W(u, v) = \max\left(0.1,\ S \cdot \ln(1 + C_{\text{eff}})\right)$$

#### Variable Definitions:

* **$W(u, v)$ (Composite Weight):** The final numerical edge weight used to determine line thickness in PyVis and connection priority in the GraphRAG retriever.
* **$S$ (Source Diversity):** The number of unique document sources (`Source`) containing the connection between $u$ and $v$.
* **$C_{\text{eff}}$ (Effective Co-occurrence Count):** The sum of proximity-decayed scores across all $C$ texts in which $u$ and $v$ co-occur (one $d_k$ per text):
  $$C_{\text{eff}} = \sum_{k=1}^{C} e^{-\lambda \cdot d_k}$$
* **$d_k$ (Character Distance):** The distance in characters between the end offset of entity 1 and the start offset of entity 2 in occurrence $k$, measured on their closest mentions in the text (the closest pair inside one sentence when there is one).
* **$\lambda$ (Decay Rate):** Hyperparameter ($\lambda = 0.01$) controlling how rapidly connection strength decays with text distance.
* **$\text{Avg Proximity}$ (Average Proximity Score):** A normalized metric ($\text{Avg Proximity} = C_{\text{eff}} / C \in (0, 1]$) representing how close $u$ and $v$ appear to each other on average whenever co-occurring.

---

#### Rationale & Design Logic:

1. **Exponential Distance Decay ($e^{-\lambda \cdot d_k}$):**
   * Entities mentioned side-by-side in the same phrase (e.g., $d_k = 0 \implies e^0 = 1.0$) are far more likely to share a direct relation than entities separated by hundreds of characters in a long paragraph.
   * Applying exponential decay scales the value of each co-occurrence according to textual proximity.

2. **Strictly Positive Logarithmic Scaling ($\ln(1 + C_{\text{eff}})$):**
   * Using a natural logarithm enforces *diminishing returns* on repetition, preventing a single verbose document from artificially inflating an edge weight.
   * Using $\ln(1 + C_{\text{eff}})$ (or `math.log1p`) guarantees that any $C_{\text{eff}} > 0$ yields a strictly positive weight ($W > 0$), eliminating negative weights caused by small proximity scores.

3. **Linear Multiplier for Source Verification ($S$):**
   * Cross-validation across distinct document sources is the strongest indicator of factual reliability in GraphRAG.
   * Multiplying by $S$ heavily rewards connections corroborated across multiple independent articles.

---

#### Graph Schema:

* **Nodes**
  * `PER` / `ORG` / `LOC` entities (`type`); `LOC` nodes also hold `latitude`, `longitude`, `address`, `place_type`, `importance`, `geometry_wkt` (WGS 84, EPSG:4326).
  * `TEXT` nodes (`TEXT_<text_id>`): `text_id` (0..N-1, original order), `text`, `source`, `index`, `title`, `sub_title`.
* **Entity–entity edges:** `source`, `target`, `weight` ($W$ above) and `relations`, a dictionary of relation → list of occurrences:
  `{"text_id", "source_start", "source_end", "target_start", "target_end", "subject", "confidence"}`.
  An occurrence is the pair in one sentence that passes the dual gate (each sentence is classified separately), or, when no sentence of the text passes it, the pair in the text (`CO_OCCURS_WITH`). Two items of one comma list (only commas between them) are `CO_OCCURS_WITH` without von. $C_{\text{eff}}$ still counts one $d_k$ per text.
  Spans are character offsets of the source/target entity in the text node; `subject` is the entity the relation reads from (`null` = symmetric); `confidence` is von's score (`null` when the pair failed the dual gate).
* **`IN_THE_TEXT` edges:** entity → text node, `weight` = number of mentions, `relations = {"IN_THE_TEXT": [{"start", "end"}, ...]}`.
* **`INSIDE` relations:** location → smallest location whose polygon contains its point, `{"subject": <inner location>, "basis": "geometry"}`; without a text connection the edge weight is the floor $W = 0.1$ ($S = 0$).


##Exporting the Graph

In [ ]:
# 5. Export to RAG-compatible JSON, compact and gzip-compressed (~37 MB), into the repo folder
import gzip

KG_PATH = 'Corpus2GeoRAG/Graph_n_Geocoding/knowledge_graph.json.gz'
graph_data = nx.node_link_data(G, edges="links")

with gzip.open(KG_PATH, 'wt', encoding='utf-8') as f:
    json.dump(graph_data, f, ensure_ascii=False, separators=(',', ':'))

print(f"Successfully exported '{KG_PATH}' ({os.path.getsize(KG_PATH) / 1e6:.0f} MB): commit and push it to GitHub.")


In [ ]:
# # ==========================================
# # 1. BUILD & EXPORT EDGES DATAFRAME
# # ==========================================
# edge_records = []

# for u, v, data in G.edges(data=True):
#     # Join list fields with a pipe '|' delimiter for clean CSV cell formatting
#     relations_str = "|".join(data.get('relations', []))
#     sources_str = "|".join(data.get('sources', []))
#     evidences_list = data.get('evidences', [])
#     primary_evidence = evidences_list[0] if evidences_list else ""

#     edge_records.append({
#         'source': u,
#         'target': v,
#         'source_type': G.nodes[u].get('type', ''),
#         'target_type': G.nodes[v].get('type', ''),
#         'weight': data.get('weight', 0.1),
#         'relations': relations_str,
#         'raw_occurrences': data.get('raw_occurrences', 0),
#         'effective_cooccurrences': round(data.get('c_eff', 0.0), 3),
#         'num_sources': len(data.get('sources', [])),
#         'sources': sources_str,
#         'primary_evidence': primary_evidence
#     })

# edges_df = pd.DataFrame(edge_records)

# # Sort by weight descending for easier inspection
# edges_df = edges_df.sort_values(by='weight', ascending=False).reset_index(drop=True)

# # Save to CSV using 'utf-8-sig' to ensure correct non-ASCII character encoding (e.g., Cyrillic/Hebrew in Excel)
# edges_df.to_csv('knowledge_graph_edges.csv', index=False, encoding='utf-8-sig')


# # ==========================================
# # 2. BUILD & EXPORT NODES DATAFRAME
# # ==========================================
# node_records = []

# for node, data in G.nodes(data=True):
#     node_records.append({
#         'entity': node,
#         'type': data.get('type', ''),
#         'latitude': data.get('latitude', None),
#         'longitude': data.get('longitude', None),
#         'address': data.get('address', None)
#     })

# nodes_df = pd.DataFrame(node_records)
# nodes_df.to_csv('knowledge_graph_nodes.csv', index=False, encoding='utf-8-sig')

# print(f"Exported {len(edges_df)} edges to 'knowledge_graph_edges.csv'")
# print(f"Exported {len(nodes_df)} nodes to 'knowledge_graph_nodes.csv'")



##Visulazing

In [ ]:
import IPython
from pyvis.network import Network

# The full graph is too large for the browser: draw the top entities by weighted degree (text nodes excluded)
TOP_N_ENTITIES = 150

entity_graph = G.subgraph(n for n, t in G.nodes(data='type') if t != 'TEXT')
top_nodes = sorted(entity_graph.degree(weight='weight'), key=lambda x: x[1], reverse=True)[:TOP_N_ENTITIES]
H = entity_graph.subgraph(n for n, _ in top_nodes)

# Initialize PyVis network (directed=False for undirected edges)
net = Network(
    notebook=True,
    cdn_resources='in_line',
    height='650px',
    width='100%',
    bgcolor='#1e1e1e',
    font_color='white',
    directed=False
)

color_map = {'PER': '#42a5f5', 'LOC': '#66bb6a', 'ORG': '#ffa726'}
pos = nx.spring_layout(H, seed=42, k=0.5)

# Add Nodes
for node, data in H.nodes(data=True):
    group_type = data.get('type', 'ORG')
    x_pos = float(pos[node][0]) * 1000
    y_pos = float(pos[node][1]) * 1000
    address = f"<br>Address: {data['address']}" if data.get('address') else ''

    net.add_node(
        node,
        label=node,
        color=color_map.get(group_type, '#cccccc'),
        x=x_pos,
        y=y_pos,
        title=f"Entity: {node}<br>Type: {group_type}{address}"
    )

# Add Edges
for u, v, data in H.edges(data=True):
    w = data.get('weight', 1.0)
    relations_str = ', '.join(f'{rel} ({len(occ)})' for rel, occ in data['relations'].items())

    edge_tooltip = (
        f"<b>Edge:</b> {u} &mdash; {v}<br>"
        f"<b>Relations:</b> {relations_str}<br>"
        f"<b>Composite Weight (W):</b> {w}"
    )

    net.add_edge(
        u, v,
        value=w,
        title=edge_tooltip
    )

# Freeze layout
net.set_options("""
var options = {
  "physics": {
    "enabled": true
  },
  "interaction": {
    "dragNodes": true,
    "zoomView": true
  }
}
""")

net.show('undirected_rag_graph.html')
IPython.display.HTML(filename='undirected_rag_graph.html')


##Metrics

In [ ]:
# Metrics are computed on the entity graph (text nodes and IN_THE_TEXT edges excluded)
entity_graph = G.subgraph(n for n, t in G.nodes(data='type') if t != 'TEXT').copy()

# Calculate degree centrality for all nodes in the entity graph
degrees = dict(entity_graph.degree())

# Sort entities by degree in descending order and get the top 10
top_10_energy_entities = sorted(degrees.items(), key=lambda item: item[1], reverse=True)[:10]

print("Top 10 highest energy (degree connectivity) entities:")
for idx, (entity, degree) in enumerate(top_10_energy_entities, 1):
    node_type = G.nodes[entity].get('type', 'Unknown')
    print(f"{idx}. {entity} ({node_type}) - Degree: {degree}")
